# Sustainable HR â€” Quick Analysis Notebook

This lightweight notebook demonstrates the common analysis workflow: load data, validate, preprocess, feature-engineer, train a baseline model, run cross-validation, save predictions, and produce diagnostic visualisations. Use the `scratch_models` folder for disposable outputs.

## Imports

In [ ]:
import pathlib
import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from src import data_loader, validation, preprocessing, features, modelling, evaluation, evaluation_visuals, targets, utils

OUT_DIR = pathlib.Path('scratch_models')
OUT_DIR.mkdir(exist_ok=True)

## Load data

In [ ]:
# Load all CSVs in data/raw (or pass a single file path)
df = data_loader.load_all_data('data/raw')
print(df.shape)
df.head()

## Validate data (quick checks)

In [ ]:
df_valid, report = validation.validate_and_coerce(df)
print('Validation report:')
print(report)
# Save report for reference
import json
(OUT_DIR / 'validation_report.json').write_text(json.dumps(report, indent=2))
df_valid.head()

## Preprocess & feature engineering

In [ ]:
df_proc = preprocessing.preprocess_data(df_valid, drop_na_columns=['distance_km'])
df_feat = features.compute_features(df_proc)
print(df_feat.shape)
df_feat.head()

## Select features and train a baseline model

In [ ]:
target = 'avg_hr'
feature_cols = [c for c in df_feat.select_dtypes(include=['number']).columns if c != target and not c.startswith('predicted_')]
print('Feature columns:', feature_cols)
# Fit a simple linear baseline and attach predictions to the dataframe
model, df_pred = modelling.fit_sustainable_hr_model(df_feat, features=feature_cols, target=target, model_file=str(OUT_DIR / 'hr_model.joblib'))
metrics = evaluation.evaluate_model(df_pred[target], df_pred['predicted_sustainable_hr'])
print('Training metrics:')
print(metrics)
df_pred.head()

## Cross-validation (quick)

In [ ]:
cv_est = LinearRegression()
cv_results = evaluation.cross_validate_model_from_df(cv_est, df_feat, feature_cols, target, cv=5)
print('CV summary:')
print({k: cv_results[k] for k in ['mae_mean','mae_std','r2_mean','r2_std']})
# Save CV results and a fold error plot
import json
(OUT_DIR / 'cv_hr_results.json').write_text(json.dumps(cv_results, indent=2))
evaluation_visuals.plot_cv_metrics(cv_results, out_path=str(OUT_DIR / 'cv_fold_errors.png'))
print('CV fold errors plot saved')

## Save predictions and visualisations

In [ ]:
# Save predictions for inspection
df_pred.to_csv(OUT_DIR / 'predictions.csv', index=False)
# Diagnostic plots
evaluation_visuals.plot_predicted_vs_actual(df_pred[target], df_pred['predicted_sustainable_hr'], out_path=str(OUT_DIR / 'predicted_vs_actual.png'))
evaluation_visuals.plot_residuals_distribution(df_pred[target], df_pred['predicted_sustainable_hr'], out_path=str(OUT_DIR / 'residuals_distribution.png'))
# Feature importance (linear model)
try:
    evaluation_visuals.plot_feature_importance(model, feature_cols, out_path=str(OUT_DIR / 'feature_importance.png'))
except Exception as e:
    print('Feature importance plot skipped:', e)
print('Saved predictions and plots to', OUT_DIR)

## Scenario prediction example

In [ ]:
# Build a scenario using dataset medians and predict HR for a 10 km run at 5 min/km
scenario = targets.build_scenario_template(df_feat, pace_min_km=5.0, distance_km=10.0)
scenario_proc = preprocessing.preprocess_data(scenario, compute_features=True)
scenario_feat = features.compute_features(scenario_proc)
scenario_features = [c for c in scenario_feat.select_dtypes(include=['number']).columns if c not in ['avg_hr'] and not c.startswith('predicted_')]
scenario_pred = model.predict(scenario_feat[scenario_features])
scenario_feat['predicted_sustainable_hr'] = scenario_pred
scenario_feat.to_dict(orient='records')

## Training history

In [ ]:
# Rolling training load added by feature pipeline
evaluation_visuals.plot_training_history(df_feat, metric="distance_km")
if "rolling_7run_load" in df_feat.columns:
    evaluation_visuals.plot_training_history(df_feat, metric="rolling_7run_load")

## Train inverse pace model

In [ ]:
# Train a pace model: given HR + distance → predict pace
pace_feature_cols = [c for c in df_feat.select_dtypes(include=["number"]).columns
                     if c != "avg_pace_min_km" and not c.startswith("predicted_")]
pace_model, df_pace = modelling.fit_pace_from_hr_model(
    df_feat, features=pace_feature_cols, model_file=str(OUT_DIR / "pace_model.joblib")
)
pace_metrics = evaluation.evaluate_model(df_pace["avg_pace_min_km"], df_pace["predicted_pace_min_km"])
print("Pace model metrics:", pace_metrics)

## Race prediction

Use the trained HR and pace models to predict sustainable effort across standard race distances.

- **Forward**: at your typical training pace, what HR does the model predict for each race?
- **Inverse**: given a target HR, what pace (and finish time) can you sustain?
- **Target time**: given a desired finish time, what pace is required?

In [ ]:
from src import race_predictor

# Full report: predicted HR at median pace, plus sustainable pace at target HR
MY_TARGET_HR = 155  # adjust to your aerobic threshold / race HR goal

report = race_predictor.race_report(
    model, pace_model, df_feat,
    feature_cols_hr=feature_cols,
    feature_cols_pace=pace_feature_cols,
    target_hr=MY_TARGET_HR,
)
print(report.to_string(index=False))
report.to_csv(OUT_DIR / "race_predictions.csv", index=False)

In [ ]:
# What pace and finish time does a 3:30 marathon target require?
result = race_predictor.predict_for_target_time(
    race="marathon", target_time_min=210
)
print(result)

In [ ]:
# Predict HR at a specific pace for each race (e.g., 5 min/km tempo pace)
for race in race_predictor.RACE_DISTANCES_KM:
    r = race_predictor.predict_hr_for_race(model, df_feat, race, pace_min_km=5.0, feature_cols=feature_cols)
    print(f"{r['race']:15s}  HR: {r['predicted_hr_bpm']} bpm  Finish: {r['estimated_finish_time']}")

## Next steps

- Drop your real Garmin `Activities.csv` into `data/raw/` and re-run this notebook.
- Try regularised or tree-based models (`Ridge`, `RandomForest`) for better generalisation.
- Add heart-rate zone features (time-in-zone) once lap-level data is available.
- Build a Streamlit or Dash dashboard around `dashboard.py` for an interactive view.